# Using an AI assistant for Python, with checks

Python stage, lesson P27. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** write a prompt with the columns and the grain, read a drafted function before running it, test it on a tiny frame you worked by hand, and reconcile it on the real data with a second route.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import linecache
import sys
import traceback

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def show_error(code, short=False):
    """Run a few lines that fail on purpose and print the error, so Run all keeps going."""
    code = code.strip("\n") + "\n"
    linecache.cache["<cell>"] = (len(code), None, code.splitlines(True), "<cell>")
    try:
        exec(compile(code, "<cell>", "exec"), globals())
    except Exception as error:
        if short:
            print("".join(traceback.format_exception_only(type(error), error)).strip().split("\n")[-1])
        else:
            traceback.print_exception(type(error), error, error.__traceback__.tb_next, file=sys.stdout)


def check_function(label, name, cases):
    """Call your function on every case. Each case is (arguments, keyword arguments, expected result)."""
    fn = globals().get(name)
    if not callable(fn):
        RESULTS[label] = None
        print(f"{label}: define a function called {name}, then run this cell again.")
        return False
    for args, kwargs, expected in cases:
        try:
            got = fn(*args, **kwargs)
        except Exception as error:
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} stopped with {type(error).__name__}: {error}")
            return False
        if not same(got, expected):
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} should give {expected!r}, but gave {got!r}.")
            if got is None:
                print("Hint: None means the function gave nothing back. Did it return the value, or only print it?")
            return False
    RESULTS[label] = True
    print(f"{label}: correct ({len(cases)} cases).")
    return True

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each order line, as built in lessons P13 and P15: the order, the store name, the month, the category and the amount.
price = dict(zip(products["product_id"], products["unit_price_usd"]))
names = dict(zip(stores["store_id"], stores["store_name"]))
kind = dict(zip(products["product_id"], products["category"]))
lines = order_items.merge(orders[["order_id", "store_id", "ordered_at"]], on="order_id", validate="many_to_one")
lines = lines.assign(store=lines["store_id"].map(names), category=lines["product_id"].map(kind),
                     month=pd.to_datetime(lines["ordered_at"]).dt.month,
                     amount=lines["quantity"] * lines["product_id"].map(price) - lines["discount_usd"])
lines = lines[["order_id", "store", "month", "category", "amount"]]

## Unit 1: Ask with the columns, the grain and a few rows

**Predict first.** the table has 3,287 rows. How many lines long will the prompt be?

In [ ]:
sample = lines.head(3).to_string(index=False)
prompt = (
    "I have a pandas DataFrame called lines: one row for each order line of a cafe chain, "
    f"with the columns {', '.join(lines.columns)}. The amount is in US dollars.\n"
    f"Here are the first three rows:\n{sample}\n"
    "Write a function coffee_share(lines) that returns, for each store, "
    "the share of that store's sales that came from coffee."
)
print(prompt)
print(len(prompt.splitlines()), "lines;", len(lines), "rows in the table")

## Unit 2: Read the draft before you run it

**Predict first.** will it run without an error, and how many numbers will it print?

In [ ]:
def coffee_share(lines):
    coffee = lines[lines["category"] == "coffee"]
    return coffee.groupby("store")["amount"].sum() / lines["amount"].sum()


print(coffee_share(lines).round(3).to_dict())

## Unit 3: Test it on a frame you worked by hand

**Predict first.** what will the draft give for each store on `tiny`, and will the assert stop?

In [ ]:
tiny = lines.iloc[[0, 1, 2, 3, 4, 9]]
hand = {"Campus": 0.768, "Old Town": 0.275}  # coffee over all sales of the same store, by hand
got = coffee_share(tiny).round(3).to_dict()
print(got)
show_error('assert got == hand, f"hand {hand}, draft {got}"', short=True)

## Unit 4: Reconcile on the real data and keep a note

**Predict first.** will either assert stop the cell, and what will the three shares be?

In [ ]:
def coffee_share(lines):
    coffee = lines[lines["category"] == "coffee"]
    return coffee.groupby("store")["amount"].sum() / lines.groupby("store")["amount"].sum()


line_share = lines["amount"] / lines.groupby("store")["amount"].transform("sum")
second = line_share[lines["category"] == "coffee"].groupby(lines["store"]).sum()
assert coffee_share(tiny).round(3).to_dict() == hand, "the tiny frame disagrees"
assert coffee_share(lines).round(6).to_dict() == second.round(6).to_dict(), "the second route disagrees"
print(coffee_share(lines).round(3).to_dict())
note = {"asked": "coffee share of each store's sales", "date": "2026-11-20", "draft": "illustrative, written for the lesson",
        "checks": "tiny frame worked by hand; second route with transform", "rows": f"all {len(lines):,} order lines"}
for key, value in note.items():
    print(f"{key}: {value}")

## Finish it

### Exercise 1: Finish it

**Task:** A drafted function for the average number of lines in an order printed a number. Is it right? The cell below holds the draft (illustrative, written for this lesson) and three tiny frames of real lines: `one` is order 1001 alone, `five` is orders 1001 and 1002, `four` is orders 1003 to 1005. Work out the answer for `five` by hand first, then run the cell and compare. Finish it: fix the function so that it gives the number of lines divided by the number of different orders for every frame.

**Expected:** `lines_per_order(five)` is the number you worked out by hand.

In [ ]:
def lines_per_order(frame):  # the assistant's draft (illustrative)
    return len(frame) / frame["order_id"].count()


one = lines.iloc[[0, 1, 2]]
five = lines.iloc[[0, 1, 2, 3, 4]]
four = lines.iloc[[5, 6, 7, 8]]
print(lines_per_order(five))  # compare with your hand answer

In [ ]:
_ = check_function("Exercise 1", "lines_per_order", [((five,), {}, 2.5), ((one,), {}, 3.0), ((four,), {}, 4 / 3)])

<details>
<summary>Hint</summary>

`count()` counts the rows that have a value in the column, and every row has an order id. Which method counts different values?

</details>

<details>
<summary>Solution</summary>

```python
def lines_per_order(frame):
    return len(frame) / frame["order_id"].nunique()


one = lines.iloc[[0, 1, 2]]
five = lines.iloc[[0, 1, 2, 3, 4]]
four = lines.iloc[[5, 6, 7, 8]]
print(lines_per_order(five))
```

`five` holds 5 lines of 2 orders, so 2.5; `one` is 3 lines of 1 order, 3.0; `four` is 4 lines of 3 orders. The draft divided the rows by the rows that have an order id, which is every row, and gave 1.0 for each frame, so its number never changed whatever the orders. Grouping by order and adding the group sizes gives the row count again, also 1.0, and the check cell calls both.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many orders were placed from May 15 through June 15, both days included? An illustrative draft, written for this lesson, answers the same question with `orders[(orders["ordered_at"] >= "2026-05-15") & (orders["ordered_at"] <= "2026-06-15")]`. `ordered_at` holds text such as `2026-06-15 09:44:00`. Before you start, say whether you expect the draft to include the orders of June 15. Then write your own count in the empty cell of the notebook and set `answer_2` to the number.

**Expected:** `answer_2` is a whole number.

In [ ]:
_ = check("Exercise 2", "answer_2", 388, traps={377: "That is the draft's count. Does the text `2026-06-15 09:44:00` come before or after the text `2026-06-15`?"})

<details>
<summary>Hint</summary>

Lesson P17 turns the text into timestamps. A timestamp at 09:44 on June 15 is after the start of June 15, so ask for everything before the start of June 16.

</details>

<details>
<summary>Solution</summary>

```python
ts = pd.to_datetime(orders["ordered_at"])
answer_2 = int(((ts >= "2026-05-15") & (ts < "2026-06-16")).sum())
```

388 orders. The draft gave 377 because the text `2026-06-15 09:44:00` sorts after the text `2026-06-15`, so every order of June 15, 11 of them, fell outside `<=`. Turning the text into timestamps and stopping before June 16 includes the whole day. A tiny frame with one order at 09:44 on June 15 would have caught the draft.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. An illustrative draft, written for this lesson, prints the growth of the sales from each month to the next, from February to June. The owner expected one number for each step in calendar order. Fix the lines so that `answer_3` holds the five growth rates in calendar order, each rounded to 4 decimals.

**Expected:** `answer_3` is a list of five numbers.

In [ ]:
month_name = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun"}
monthly = lines.assign(name=lines["month"].map(month_name)).groupby("name")["amount"].sum()
answer_3 = monthly.pct_change().round(4).dropna().tolist()  # the draft's growth, February to June

In [ ]:
_ = check("Exercise 3", "answer_3", [-0.1413, 0.1236, 0.4635, 0.1623, -0.1221], traps={(-0.3918, 0.1646, 0.4406, -0.3303, 0.7009): "Print `monthly`. In which order are the months, and what does `pct_change` compare each row with?"})

<details>
<summary>Hint</summary>

Print `monthly` and read the order of its rows. `pct_change` compares each row with the row above it.

</details>

<details>
<summary>Solution</summary>

```python
monthly = lines.groupby("month")["amount"].sum()
answer_3 = monthly.pct_change().round(4).dropna().tolist()
```

The five rates are -0.1413, 0.1236, 0.4635, 0.1623 and -0.1221: the sales fell in February, rose in March, April and May, and fell in June. The draft grouped by the month names, and `groupby` sorts text, so the rows ran Apr, Feb, Jan, Jun, Mar, May and each rate compared a month with the one before it in the alphabet. Grouping by the month number keeps calendar order. Six rows gave five rates either way, so the shape never gave it away.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")